# Reto: Web Scraping + API + Base de datos
## books.toscrape.com
- Scraping completo: 1000 libros, 50 categorías
- API: Open Library
- DB: SQLite

In [1]:
# Librerías necesarias para el proyecto
import requests
from bs4 import BeautifulSoup
import sqlite3
import json
import time

URL_BASE   = "https://books.toscrape.com/"
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}  # convierte rating de texto a número
cache      = {}  # evita llamadas duplicadas a la API


In [2]:
# Entra a la página de detalle de un libro y extrae UPC, disponibilidad y descripción
def scrape_detalle(url):
    soup = BeautifulSoup(requests.get(url).text, "html.parser")
    
    # La tabla de producto tiene filas th (nombre) → td (valor), las convertimos a diccionario
    datos = {}
    for fila in soup.select("table.table-striped tr"):
        clave = fila.find("th").text.strip()
        valor = fila.find("td").text.strip()
        datos[clave] = valor

    # La descripción está en el <p> que sigue al div#product_description
    desc_tag = soup.find("div", id="product_description")
    descripcion = desc_tag.find_next_sibling("p").text.strip() if desc_tag else ""

    return {
        "upc":            datos.get("UPC"),
        "disponibilidad": datos.get("Availability"),
        "descripcion":    descripcion
    }


In [3]:
# Consulta Open Library por el título del libro y devuelve datos del autor
def buscar_autor(titulo):
    if titulo in cache:
        return cache[titulo]
    try:
        datos = requests.get(
            f"https://openlibrary.org/search.json?title={titulo.replace(' ', '+')}",
            timeout=8
        ).json()

        if not datos["docs"]:
            cache[titulo] = {}
            return {}

        primer = datos["docs"][0]
        key    = primer.get("author_key", [None])[0]

        # Segunda llamada: datos del autor
        datos_autor = requests.get(
            f"https://openlibrary.org/authors/{key}.json", timeout=8
        ).json() if key else {}

        # Extraer año de nacimiento desde birth_date
        birth_date = datos_autor.get("birth_date", "")
        anio = next((p for p in birth_date.split() if p.isdigit() and len(p) == 4), None)
        if anio:
            anio = int(anio)

        # Intentar obtener país desde birth_place primero
        pais = datos_autor.get("birth_place")

        # Si no hay birth_place, buscar en la bio
        if not pais:
            bio = datos_autor.get("bio", {})
            if isinstance(bio, dict):
                bio = bio.get("value", "")
            nacionalidades = {
                "British":    "United Kingdom",
                "American":   "United States",
                "Canadian":   "Canada",
                "Australian": "Australia",
                "Irish":      "Ireland",
                "Scottish":   "Scotland",
            }
            for palabra, pais_nombre in nacionalidades.items():
                if palabra.lower() in bio.lower():
                    pais = pais_nombre
                    break

        # Tercera llamada: cantidad de obras del autor
        works_count = None
        if key:
            works_data = requests.get(
                f"https://openlibrary.org/authors/{key}/works.json",
                timeout=8
            ).json()
            works_count = works_data.get("size", 0)

        resultado = {
            "autor":             primer.get("author_name", [None])[0],
            "anio":              anio,
            "pais":              pais,
            "id_externo":        key,
            "total_known_works": works_count
        }
        cache[titulo] = resultado
        return resultado
    except:
        cache[titulo] = {}
        return {}

In [7]:
# Scraping completo: recorre las 50 categorías y todos sus libros
todos_los_libros = []

soup_principal = BeautifulSoup(requests.get(URL_BASE).text, "html.parser")
categorias = soup_principal.select(".side_categories ul li ul li a")

for cat in categorias:
    nombre_categoria = cat.text.strip()
    url_actual = URL_BASE + cat["href"]

    # Paginación: sigue hasta que no haya botón 'next'
    while True:
        soup = BeautifulSoup(requests.get(url_actual).text, "html.parser")

        for libro in soup.find_all("article", class_="product_pod"):
            titulo    = libro.find("h3").find("a")["title"]
            precio    = float(libro.find("p", class_="price_color").text.strip().replace("Â", "").replace("£", ""))
            rating    = RATING_MAP[libro.find("p", class_="star-rating")["class"][1]]
            url_libro = URL_BASE + "catalogue/" + libro.find("h3").find("a")["href"].replace("../", "")

            detalle     = scrape_detalle(url_libro)
            datos_autor = buscar_autor(titulo)

            todos_los_libros.append({
                "titulo":         titulo,
                "precio":         precio,
                "rating":         rating,
                "categoria":      nombre_categoria,
                "upc":            detalle.get("upc"),
                "disponibilidad": detalle.get("disponibilidad"),
                "autor":          datos_autor.get("autor"),
                "anio":           datos_autor.get("anio"),
                "pais":           datos_autor.get("pais"),
                "id_externo":     datos_autor.get("id_externo"),
                "total_known_works":  datos_autor.get("total_known_works")
            })

        siguiente = soup.find("li", class_="next")
        if siguiente:
            href = siguiente.find("a")["href"]
            url_actual = url_actual.rsplit("/", 1)[0] + "/" + href
        else:
            break

        time.sleep(0.3)

    print(f"{nombre_categoria} listo")

print(f"Total: {len(todos_los_libros)} libros")
print(todos_los_libros[0])


Travel listo
Mystery listo
Historical Fiction listo
Sequential Art listo
Classics listo
Philosophy listo
Romance listo
Womens Fiction listo
Fiction listo
Childrens listo
Religion listo
Nonfiction listo
Music listo
Default listo
Science Fiction listo
Sports and Games listo
Add a comment listo
Fantasy listo
New Adult listo
Young Adult listo
Science listo
Poetry listo
Paranormal listo
Art listo
Psychology listo
Autobiography listo
Parenting listo
Adult Fiction listo
Humor listo
Horror listo
History listo
Food and Drink listo
Christian Fiction listo
Business listo
Biography listo
Thriller listo
Contemporary listo
Spirituality listo
Academic listo
Self Help listo
Historical listo
Christian listo
Suspense listo
Short Stories listo
Novels listo
Health listo
Politics listo
Cultural listo
Erotica listo
Crime listo
Total: 1000 libros
{'titulo': "It's Only the Himalayas", 'precio': 45.17, 'rating': 2, 'categoria': 'Travel', 'upc': 'a22124811bfa8350', 'disponibilidad': 'In stock (19 available)', '

In [8]:
# Guardar los datos scrapeados en disco
with open("libros_backup.json", "w", encoding="utf-8") as f:
    json.dump(todos_los_libros, f, ensure_ascii=False, indent=2)

print(f"Guardados {len(todos_los_libros)} libros")
print(f"Última categoría: {todos_los_libros[-1]['categoria']}")


Guardados 1000 libros
Última categoría: Crime


In [9]:
# Cargar los datos desde el backup
with open("libros_backup.json", "r", encoding="utf-8") as f:
    todos_los_libros = json.load(f)

print(f"Cargados {len(todos_los_libros)} libros")


Cargados 1000 libros


In [10]:
# Crear la base de datos con las 4 tablas
# Primero las tablas sin FK, después las que dependen de ellas
conexion = sqlite3.connect("libros.db")
cursor  = conexion.cursor()

cursor.executescript("""
    CREATE TABLE IF NOT EXISTS categorias (
        id     INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT NOT NULL UNIQUE
    );

    CREATE TABLE IF NOT EXISTS autores (
        id                INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre            TEXT NOT NULL UNIQUE,
        anio              INTEGER,
        pais              TEXT,
        id_externo        TEXT,
        total_known_works INTEGER,
        api_source        TEXT DEFAULT 'open_library',
        created_at        TEXT DEFAULT (datetime('now'))
    );

    CREATE TABLE IF NOT EXISTS libros (
        id             INTEGER PRIMARY KEY AUTOINCREMENT,
        titulo         TEXT NOT NULL,
        precio         REAL,
        rating         INTEGER,
        upc            TEXT UNIQUE,
        disponibilidad TEXT,
        categoria_id   INTEGER REFERENCES categorias(id)
    );

    CREATE TABLE IF NOT EXISTS libro_autor (
        libro_id  INTEGER REFERENCES libros(id),
        autor_id  INTEGER REFERENCES autores(id),
        PRIMARY KEY (libro_id, autor_id)
    );
""")

conexion.commit()
conexion.close()
print("Base de datos creada")


Base de datos creada


In [11]:
# Insertar los datos en la DB respetando el orden de dependencias
conexion = sqlite3.connect("libros.db")
cursor   = conexion.cursor()

for libro in todos_los_libros:

    # Categoría: INSERT OR IGNORE evita duplicados, después buscamos su id
    cursor.execute(
        "INSERT OR IGNORE INTO categorias (nombre) VALUES (?)",
        (libro["categoria"],)
    )
    cursor.execute(
        "SELECT id FROM categorias WHERE nombre = ?",
        (libro["categoria"],)
    )
    categoria_id = cursor.fetchone()[0]

    # Autor: solo si tiene datos de la API
    autor_id = None
    if libro["autor"]:
        
        cursor.execute(
            "INSERT OR IGNORE INTO autores (nombre, anio, pais, id_externo, total_known_works) VALUES (?, ?, ?, ?, ?)",
            (libro["autor"], libro["anio"], libro["pais"], libro["id_externo"], libro.get("total_known_works"))
        )
        cursor.execute(
            "SELECT id FROM autores WHERE nombre = ?",
            (libro["autor"],)
        )
        autor_id = cursor.fetchone()[0]

    # Libro: usamos el upc como identificador único
    cursor.execute("""
        INSERT OR IGNORE INTO libros (titulo, precio, rating, upc, disponibilidad, categoria_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (libro["titulo"], libro["precio"], libro["rating"],
          libro["upc"], libro["disponibilidad"], categoria_id))
    
    cursor.execute("SELECT id FROM libros WHERE upc = ?", (libro["upc"],))
    libro_id = cursor.fetchone()[0]

    # Relación libro-autor en la tabla intermedia
    if autor_id:
        cursor.execute(
            "INSERT OR IGNORE INTO libro_autor (libro_id, autor_id) VALUES (?, ?)",
            (libro_id, autor_id)
        )

conexion.commit()
conexion.close()
print("Datos insertados correctamente")


Datos insertados correctamente


In [12]:
# Verificar que los datos se insertaron correctamente
conexion = sqlite3.connect("libros.db")
cursor   = conexion.cursor()

cursor.execute("SELECT COUNT(*) FROM libros")
print("Libros:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM autores")
print("Autores:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM categorias")
print("Categorias:", cursor.fetchone()[0])

conexion.close()


Libros: 1000
Autores: 391
Categorias: 50


In [17]:
# Consulta 1: Libros bien valorados a buen precio
# Busca libros con rating máximo (5 estrellas) y precio menor a £15
# Útil para encontrar las mejores opciones calidad-precio del sitio
conexion = sqlite3.connect("libros.db")
cursor   = conexion.cursor()

cursor.execute("""
    SELECT l.titulo, l.precio, l.rating, c.nombre
    FROM   libros l
    JOIN   categorias c ON l.categoria_id = c.id
    WHERE  l.rating > 4 AND l.precio < 15
    ORDER  BY l.precio ASC
    LIMIT 10
""")

for fila in cursor.fetchall():
    print(fila)
conexion.close()


('An Abundance of Katherines', 10.0, 5, 'Young Adult')
('Greek Mythic History', 10.23, 5, 'Default')
('The Power Greens Cookbook: 140 Delicious Superfood Recipes', 11.05, 5, 'Food and Drink')
('Dear Mr. Knightley', 11.21, 5, 'Fiction')
('The Darkest Corners', 11.33, 5, 'Young Adult')
('Naturally Lean: 125 Nourishing Gluten-Free, Plant-Based Recipes--All Under 300 Calories', 11.38, 5, 'Food and Drink')
('Fruits Basket, Vol. 2 (Fruits Basket #2)', 11.64, 5, 'Sequential Art')
('Old School (Diary of a Wimpy Kid #10)', 11.83, 5, 'Humor')
('Superman Vol. 1: Before Truth (Superman by Gene Luen Yang #1)', 11.89, 5, 'Sequential Art')
('Every Heart a Doorway (Every Heart A Doorway #1)', 12.16, 5, 'Fantasy')


In [20]:
# Consulta 2: Categorías ordenadas por precio promedio
# Agrupa todos los libros por categoría y calcula el precio medio de cada una
# ROUND() limita los decimales a 2 para mejor legibilidad
conexion = sqlite3.connect("libros.db")
cursor   = conexion.cursor()

cursor.execute("""
    SELECT   c.nombre, ROUND(AVG(l.precio), 2) AS precio_promedio
    FROM     libros l
    JOIN     categorias c ON l.categoria_id = c.id
    GROUP BY c.nombre
    ORDER BY precio_promedio DESC
    LIMIT 10
""")

for fila in cursor.fetchall():
    print(fila)
conexion.close()

('Suspense', 58.33)
('Novels', 54.81)
('Politics', 53.61)
('Health', 51.45)
('New Adult', 46.38)
('Christian', 42.5)
('Sports and Games', 41.17)
('Self Help', 40.62)
('Travel', 39.79)
('Fantasy', 39.59)


In [21]:
# Consulta 3: Autores con peor promedio de rating (mínimo 3 libros)
# Usa una subconsulta para calcular el promedio y el total de libros por autor
# El WHERE externo filtra solo autores con al menos 3 libros para que el promedio sea representativo
conexion = sqlite3.connect("libros.db")
cursor   = conexion.cursor()

cursor.execute("""
    SELECT nombre, promedio, total_libros
    FROM (
        SELECT   a.nombre,
                 ROUND(AVG(l.rating), 2) AS promedio,
                 COUNT(*) AS total_libros
        FROM     autores a
        JOIN     libro_autor la ON a.id = la.autor_id
        JOIN     libros l       ON l.id = la.libro_id
        GROUP BY a.nombre
    )
    WHERE total_libros >= 3
    ORDER BY promedio ASC
    LIMIT 10
""")

for fila in cursor.fetchall():
    print(fila)
conexion.close()

('Douglas Adams', 2.0, 3)
('Jane Austen', 2.33, 3)
('Sophie Kinsella', 2.5, 6)
('Worth Books', 2.63, 8)
('David Levithan', 2.67, 3)
('George R. R. Martin', 2.75, 4)
('Whizbooks', 2.75, 4)
('J. K. Rowling', 2.88, 8)
('Stephen King', 3.0, 8)
('Gillian Flynn', 3.33, 3)


In [22]:
# Consulta 4: Ranking de autores con más libros usando función de ventana
# RANK() asigna una posición a cada autor según su cantidad de libros
# A diferencia de LIMIT, la función de ventana permite ver el número de posición real
conexion = sqlite3.connect("libros.db")
cursor   = conexion.cursor()

cursor.execute("""
    SELECT nombre, total_libros,
           RANK() OVER (ORDER BY total_libros DESC) AS posicion
    FROM (
        SELECT   a.nombre, COUNT(*) AS total_libros
        FROM     autores a
        JOIN     libro_autor la ON a.id = la.autor_id
        GROUP BY a.nombre
    )
    LIMIT 10
""")

for fila in cursor.fetchall():
    print(fila)
conexion.close()

('J. K. Rowling', 8, 1)
('Stephen King', 8, 1)
('Worth Books', 8, 1)
('Sophie Kinsella', 6, 4)
('Cassandra Clare', 4, 5)
('George R. R. Martin', 4, 5)
('Whizbooks', 4, 5)
('David Levithan', 3, 8)
('Douglas Adams', 3, 8)
('Gillian Flynn', 3, 8)


In [23]:
# Consulta 5 (obligatoria): País que produce más libros con rating mayor a 3
# Requiere datos de la API — autores sin país quedan excluidos
conexion = sqlite3.connect("libros.db")
cursor   = conexion.cursor()

cursor.execute("""
    SELECT   a.pais, COUNT(*) AS total_libros
    FROM     libros l
    JOIN     libro_autor la ON l.id  = la.libro_id
    JOIN     autores a      ON a.id  = la.autor_id
    WHERE    l.rating > 3
    AND      a.pais IS NOT NULL
    GROUP BY a.pais
    ORDER BY total_libros DESC
""")

for fila in cursor.fetchall():
    print(fila)


('United States', 33)
('United Kingdom', 14)
('Canada', 3)
('Ireland', 2)


In [25]:
# Indexación: comparación de performance antes y después de crear índices
import time

conexion = sqlite3.connect("libros.db")
cursor   = conexion.cursor()

# Consulta con JOIN múltiple — más costosa sin índices
consulta = """
    SELECT l.titulo, l.precio, l.rating, c.nombre, a.nombre, a.pais
    FROM   libros l
    JOIN   categorias c   ON l.categoria_id = c.id
    JOIN   libro_autor la ON l.id = la.libro_id
    JOIN   autores a      ON a.id = la.autor_id
    WHERE  l.rating > 3
    AND    l.precio > 20
    ORDER  BY l.precio DESC
"""

# Medición sin índice
t1 = time.time()
for _ in range(10000):
    cursor.execute(consulta)
    cursor.fetchall()
t2 = time.time()
print(f"Sin índice: {t2 - t1:.4f}s")

# Crear índices en las columnas más consultadas
cursor.execute("CREATE INDEX IF NOT EXISTS idx_rating      ON libros(rating)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_precio      ON libros(precio)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_categoria   ON libros(categoria_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_libro_autor ON libro_autor(libro_id)")
conexion.commit()

# Medición con índice
t1 = time.time()
for _ in range(10000):
    cursor.execute(consulta)
    cursor.fetchall()
t2 = time.time()
print(f"Con índice: {t2 - t1:.4f}s")

conexion.close()


Sin índice: 3.9283s
Con índice: 4.2169s
